In [15]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from data import prepare_data, make_dataloader
from pathlib import Path
import models as ms
import math

In [16]:
#MASK_PATH = "../modular_setup_improved/mask_2peaks.parquet" # for training on only those examples with 2 peaks
DATA_PATH = "synthetic+real_dataset_filtered.parquet"
EMBEDDING_PATH = "../modular_setup_improved/all_fm-rna_embeddings_filtered.pt"
BIN_EDGES_PATH = "bin_edges.npy"
SEED = 42
N_BINS = 50
DEVICE = "cpu"  # change to "cuda" if available

In [17]:
static_dim = 3
output_dim = 50

model_configs = {
    "glm_baseline": {
        "build": lambda: ms.GLMBaseline(23, output_dim),
        "epochs": 15,
        "lr": 1e-2,
    },
    "mean_pool_mlp": {
        "build": lambda: ms.MeanPoolMLP(static_dim, output_dim, hidden_size=64),
        "epochs": 15,
        "lr": 1e-3,
    },
    "bilstm": {
        "build": lambda: ms.DynamicHybridLSTM(
            hidden_size=64, num_layers=1, static_feature_size=static_dim,
            output_size=output_dim, mlp_hidden_size=64, bidirectional=False,
        ),
        "epochs": 45,
        "lr": 1e-3,
    },
    "bilstm_rna_fm": {
        "build": lambda: ms.DynamicEmbeddingHybridLSTM(
            hidden_size=64, 
            num_layers=1, 
            static_feature_size=static_dim,
            output_size=output_dim, 
            embedding_dim=640, # Explicitly configured for RNA-FM dense vectors
            mlp_hidden_size=64, 
            bidirectional=True,
        ),
        "epochs": 15,
        "lr": 1e-3,
    },
    "bilstm_rna_fm_with_struct": {
            "build": lambda: ms.DynamicEmbeddingHybridLSTMwithStruct(
                hidden_size=64, 
                num_layers=1, 
                struct_dim=3,  # Explicitly configured for structure features
                static_feature_size=static_dim,
                output_size=output_dim, 
                embedding_dim=640, # Explicitly configured for RNA-FM dense vectors
                mlp_hidden_size=64, 
                bidirectional=True,
            ),
            "epochs": 15,
            "lr": 1e-3,
        },
    "bilstm_rna_fm_proj": {
        "build": lambda: ms.DynamicEmbeddingHybridLSTM_proj(
            hidden_size=64, 
            num_layers=1, 
            static_feature_size=static_dim,
            output_size=output_dim, 
            embedding_dim=640, # Explicitly configured for RNA-FM dense vectors
            mlp_hidden_size=64, 
            bidirectional=True,
            projection_dim=64,  # Optional projection layer dimension
        ),
        "epochs": 15,
        "lr": 1e-3,
    },
    "embed_transformer": {
        "build": lambda: ms.Embed_Transformer(
            embedding_dim=640,
            projection_dim=64,
            num_heads=4,
            num_layers=1,
            dropout=0.5,
            static_feature_size=static_dim,
            output_size=output_dim,
            mlp_hidden_size=64
        ),
        "epochs": 15,
        "lr": 1e-3,
    },
    "rna_loc": {
        "build": lambda: ms.RNALocLM(
            embedding_dim = 640, cnn_channels = 128,
            kernel_sizes = (3, 4, 5), lstm_hidden   = 128,
            lstm_layers = 1, num_heads = 8,
            static_feature_size = 3, output_size = 50,
            mlp_hidden_size = 64, dropout = 0.3
        ),
        "epochs": 15,
        "lr": 1e-3,
    },
    "all_local_MLP": {
        "build": lambda: ms.AllLocalMLP(
            static_feature_size=static_dim, output_size=output_dim,
            hidden_size=64, num_hidden_layers=2, dropout=0.1,
        ),
        "epochs": 15,
        "lr": 1e-3,
    },
}

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [18]:
for model in model_configs:
    print(model, count_parameters(model_configs[model]["build"]()))

glm_baseline 1200
mean_pool_mlp 3954
bilstm 26290
bilstm_rna_fm 373170
bilstm_rna_fm_with_struct 374706
bilstm_rna_fm_proj 119282
embed_transformer 94898
rna_loc 1792818
all_local_MLP 7666


In [41]:
df = pd.DataFrame()

In [42]:
list_models = []
list_paramcounts = []

for model in model_configs:
    list_models.append(model)
    list_paramcounts.append(count_parameters(model_configs[model]["build"]()))

df['model'] = list_models
df['paramcount'] = list_paramcounts

In [43]:
df['seq_onehot'] = [False, True, True, False, False, False, False, False, False]
df['struct_onehot'] = [False, True, True, False, True, False, False, False, False]
#df['static_features'] = [True, True, True, True, True, True, True, True, True]
df['static_features'] = [23, 3, 3, 3, 3, 3, 3, 3, 33]
df['seq_embed'] = [False, False, False, True, True, True, True, True, True]

In [44]:
table_string = df.to_latex()

table_string = table_string.replace('True', r'$\checkmark$')
table_string = table_string.replace('False', '')
table_string = table_string.replace('_', r'\_')

In [45]:
print(table_string)

\begin{tabular}{llrrrrr}
\toprule
 & model & paramcount & seq\_onehot & struct\_onehot & static\_features & seq\_embed \\
\midrule
0 & glm\_baseline & 1200 &  &  & 23 &  \\
1 & mean\_pool\_mlp & 3954 & $\checkmark$ & $\checkmark$ & 3 &  \\
2 & bilstm & 26290 & $\checkmark$ & $\checkmark$ & 3 &  \\
3 & bilstm\_rna\_fm & 373170 &  &  & 3 & $\checkmark$ \\
4 & bilstm\_rna\_fm\_with\_struct & 374706 &  & $\checkmark$ & 3 & $\checkmark$ \\
5 & bilstm\_rna\_fm\_proj & 119282 &  &  & 3 & $\checkmark$ \\
6 & embed\_transformer & 94898 &  &  & 3 & $\checkmark$ \\
7 & rna\_loc & 1792818 &  &  & 3 & $\checkmark$ \\
8 & all\_local\_MLP & 7666 &  &  & 33 & $\checkmark$ \\
\bottomrule
\end{tabular}

